# Chapter 09: To Merge or Re-Run?
### Bayesian Decisions, Sequential Analysis & Channel Capacity

> *"To know what you know and what you do not know, that is true knowledge."*
> — Confucius

Across the first eight chapters, we developed the theoretical apparatus to
measure surprise, update beliefs, construct priors, map curvature, sample
posteriors, and compare models.

Yet in production engineering, **a posterior distribution alone cannot merge a
pull request or deploy code**.

At 4:45 PM on a Friday, an automated test suite fails on a critical hotfix pull
request. As the on-call engineer, you cannot simply gaze upon the posterior
distribution $p(\theta \mid \mathcal{D})$ and ponder its differential entropy.
You must choose one of three mutually exclusive actions:
1. **Action $a_{\text{block}}$ (Block Merge)**: Declare that the PR has
introduced a genuine bug, rejecting the commit.
2. **Action $a_{\text{merge}}$ (Force Merge)**: Declare the failure a
transient flake, overriding the test and merging to `main`.
3. **Action $a_{\text{rerun}}$ (Re-Run Test)**: Defer the binary choice,
paying compute time and developer delay to collect another sample.

Each choice carries heavy, asymmetric consequences:
- Merging a genuine bug causes a \$2,000 outage ($C_{\text{break}} = \$2{,}000$;
blocks main branch, stalls 20 engineers).
- Blocking a clean PR costs \$100 ($C_{\text{false\_block}} = \$100$;
wastes author triage time).
- Executing one more test retry costs \$1.50 ($C_{\text{run}} = \$1.50$;
cloud compute and pipeline latency).

How do we formalize this decision under uncertainty?
- In **Bayesian Statistics**, the answer is **Bayesian Decision Theory and
Wald's Sequential Probability Ratio Test (SPRT)**: minimizing expected loss and
dynamically deciding when to stop sampling.
- In **Information Theory**, the answer is **Shannon's Noisy-Channel Coding
Theorem**: test retries are error-correcting codes transmitted across an
unreliable channel.

This chapter bridges inference and action.



## 1. The Geometry of Consequence: Loss Functions and Bayes Risk

In classical statistics, a hypothesis test outputs a $p$-value or an arbitrary
rejection threshold $\alpha = 0.05$. It treats a false positive (rejecting a
working build) and a false negative (deploying a broken build to users) as
symmetric mathematical errors.

In real life, consequences are violently asymmetric.

### Operational Actions and True States
We define two true states of the universe:
- $\theta = \text{Clean}$: The commit is clean; any failure is benign noise
($\theta_0 \le 2\%$).
- $\theta = \text{Broken}$: The commit introduced a real regression
($\theta_1 \ge 40\%$).

Let $p = P(\text{Broken} \mid D)$ denote our posterior belief that the code is
broken given observed test history $D$.

We consider three operational actions:
1. **Action $a_{\text{merge}}$ (Allow Merge):**
   - Cost $= \$0$ if Clean.
   - Cost $= C_{\text{break}} = \$2{,}000$ if Broken (blocks main branch,
stalls 20 engineers).
   - Expected Bayes Risk: $R(a_{\text{merge}} \mid D) = 2000 \cdot p$.

2. **Action $a_{\text{block}}$ (Quarantine / Block PR):**
   - Cost $= C_{\text{false\_block}} = \$100$ if Clean (wastes author triage
time).
   - Cost $= \$0$ if Broken.
   - Expected Bayes Risk: $R(a_{\text{block}} \mid D) = 100 \cdot (1 - p)$.

3. **Action $a_{\text{rerun}}$ (Execute 1 More CI Run):**
   - Costs $C_{\text{run}} = \$1.50$ in cloud compute and CI latency to collect
additional evidence.

| True State $\theta$ | Action $a_{\text{merge}}$ | Action $a_{\text{block}}$ |
| :--- | :--- | :--- |
| **Clean ($\theta = 0$)** | $L(a_{\text{merge}}, \text{Clean}) = \$0$ (Correct!) | $L(a_{\text{block}}, \text{Clean}) = C_{\text{false\_block}} = \$100$ |
| **Broken ($\theta = 1$)** | $L(a_{\text{merge}}, \text{Broken}) = C_{\text{break}} = \$2{,}000$ | $L(a_{\text{block}}, \text{Broken}) = \$0$ (Correct!) |

### Optimal Posterior Threshold Derivation
The optimal Bayesian terminal decision chooses the action that minimizes
expected Bayes risk:

$$
a^* = \arg\min_a R(a \mid D)
$$

Setting the two expected risks equal yields the single-shot **Bayes-optimal
decision threshold**:

$$
R(a_{\text{merge}} \mid D) = R(a_{\text{block}} \mid D) \implies C_{\text{break}} \cdot p^* = C_{\text{false\_block}} \cdot (1 - p^*)
$$

Solving for $p^*$:

$$
p^* = \frac{C_{\text{false\_block}}}{C_{\text{break}} + C_{\text{false\_block}}} = \frac{100}{2000 + 100} = \frac{1}{21} \approx \mathbf{4.76\%}
$$

Look at this striking operational result: **Because asymmetric consequences
penalize outages 20 times more heavily than false alarms, we must block the PR
whenever $P(\text{Broken} \mid D) > 4.76\%$—not $50\%$!** A naive 'majority
vote' ($p > 0.50$) or standard scientific threshold ($\alpha = 0.05$ on null
hypothesis) is financially irresponsible.

### Bridging to Wald's SPRT: Action $a_{\text{rerun}}$ and EVSI
Now consider **Action $a_{\text{rerun}}$**: paying compute cost $C_{\text{run}}
= \$1.50$ to execute one additional test run.

Without sampling, our minimum Bayes risk is:

$$
R^*(p) = \min\left(R(a_{\text{merge}} \mid D), R(a_{\text{block}} \mid D)\right) = \min\left(2000\,p, 100\,(1 - p)\right)
$$

If we execute another run, the test will either pass or fail. The expected
reduction in Bayes risk after observing this new outcome is the **Expected Value
of Sample Information (EVSI)**:

$$
\text{EVSI}(p) = R^*(p) - \mathbb{E}_Y\left[R^*(p_Y)\right]
$$

We should choose Action $a_{\text{rerun}}$ if and only if the expected risk
reduction exceeds the test execution cost:

$$
\text{EVSI}(p) > C_{\text{run}} = \$1.50
$$

Because $R^*(p)$ is strictly concave with its peak at $p^* = 4.76\%$ (where
$R^*(p^*) = \$95.24$ and uncertainty is maximal), $\text{EVSI}(p)$ peaks at
$p^*$ and falls off as $p \to 0$ or $p \to 1$. The condition $\text{EVSI}(p) >
\$1.50$ defines an **uncertainty band** $[p_{\text{low}}, p_{\text{high}}]
\approx [0.44\%, 7.43\%]$:
- If $p < p_{\text{low}}$ ($p < 0.44\%$): EVSI is too small to justify \$1.50;
**allow merge immediately ($a_{\text{merge}}$)**.
- If $p > p_{\text{high}}$ ($p > 7.43\%$): EVSI is too small to justify \$1.50;
**quarantine/block immediately ($a_{\text{block}}$)**.
- If $p \in [p_{\text{low}}, p_{\text{high}}]$: Information gain exceeds
compute cost; **execute another test run ($a_{\text{rerun}}$)**!

This uncertainty band $[p_{\text{low}}, p_{\text{high}}]$ is the exact Bayesian
decision foundation of **Abraham Wald's Sequential Probability Ratio Test
(SPRT)**, which we formalize next in Section 2!



## 2. When to Stop Sampling
### Abraham Wald's Sequential Probability Ratio Test (SPRT)

What if after 1 test run, our posterior probability is $p = 0.04$ ($4\%$)?
- It is too high to merge ($p > 0.003$).
- It is too low to permanently block ($p < 0.50$).

We need a third action: **$a_3$ (Collect another sample)**. But running tests
costs money ($c_{\text{run}} = \$0.25$) and time. How many retries should we
execute?

Most engineering teams use a static, naive policy: *"Always retry exactly 3
times."*
- If the test was a clear regression ($k=2, n=2$), running the 3rd retry wastes
  cloud compute.
- If the test is in a razor-thin borderline state, 3 runs may leave high
  residual
  uncertainty.

In 1945, Abraham Wald invented the **Sequential Probability Ratio Test (SPRT)**,
proven by Wald and Wolfowitz to be mathematically optimal: **SPRT minimizes the
expected number of samples required to reach any specified error bounds
$(\alpha, \beta)$**.

### The SPRT Mechanics
Let $H_0: \theta = \theta_0$ (benign flakiness rate, e.g. $2\%$) and $H_1:
\theta = \theta_1$ (real bug, e.g. $40\%$). At each sequential test run $m = 1,
2, \dots$: Compute the cumulative log-likelihood ratio (the information random
walk):

$$
S_m = \sum_{i=1}^m z_i = \sum_{i=1}^m \ln \frac{p(y_i \mid H_1)}{p(y_i \mid
H_0)}
$$


Define two stopping boundaries $(B, A)$ based on acceptable error probabilities
$\alpha_{\text{err}} = P(\text{Reject } H_0 \mid H_0)$ and $\beta_{\text{err}} =
P(\text{Accept } H_0 \mid H_1)$:

$$
A = \ln \left( \frac{1 - \beta_{\text{err}}}{\alpha_{\text{err}}} \right),
\qquad B = \ln \left( \frac{\beta_{\text{err}}}{1 - \alpha_{\text{err}}} \right)
$$


At each step $m$:
- If $S_m \ge A$: **Stop and Declare Bug ($H_1$)** $\to$ Block Merge.
- If $S_m \le B$: **Stop and Declare Flake ($H_0$)** $\to$ Safe to Merge.
- If $B < S_m < A$: **Continue Testing** $\to$ Run step $m+1$.

![Wald Sequential Probability Ratio Test (SPRT) Optimal Stopping Trajectories](../images/ch09_sprt_trajectories.png)

For symmetric error thresholds $\alpha_{\text{err}} = \beta_{\text{err}} =
0.05$ (a $5\%$ false alarm and $5\%$ missed bug rate), the decision boundaries
evaluate to $A = \ln(0.95 / 0.05) = \ln(19) \approx +2.94$ and $B = \ln(0.05 /
0.95) = -\ln(19) \approx -2.94$. As illustrated above, a benign flake PR
rapidly crosses boundary $B$ in only $m=2$ runs (safe merge), a real bug crosses
boundary $A$ at $m=6$ (quarantine), and borderline flakes resolve in $m=18$
retries.



## 3. Information Geometry of SPRT: Wald's Identity & Expected Stopping Time

Why is SPRT so efficient?

Notice that each increment $z_i = \ln \frac{p(y_i \mid H_1)}{p(y_i \mid H_0)}$
has an expected drift equal to the **Kullback-Leibler Divergence**!
- Under true hypothesis $H_1$: $\mathbb{E}_{H_1}[z_i] = D_{\text{KL}}(P_{H_1}
  \parallel P_{H_0}) > 0$ (positive upward drift toward boundary $A$).
- Under true hypothesis $H_0$: $\mathbb{E}_{H_0}[z_i] = -D_{\text{KL}}(P_{H_0}
  \parallel P_{H_1}) < 0$ (negative downward drift toward boundary $B$).

By **Wald's Equation**, the expected number of test retries $\mathbb{E}[N]$
required to terminate is:

$$
\mathbb{E}_{H_1}[N] \approx \frac{(1 - \beta_{\text{err}}) A +
\beta_{\text{err}} B}{D_{\text{KL}}(P_{H_1} \parallel P_{H_0})}
$$

$$
\mathbb{E}_{H_0}[N] \approx \frac{\alpha_{\text{err}} A + (1 -
\alpha_{\text{err}}) B}{-D_{\text{KL}}(P_{H_0} \parallel P_{H_1})}
$$


**The speed of operational triage is strictly inversely proportional to the KL
divergence between the two hypotheses.** If a bug is egregious ($\theta_1 =
0.90$ vs $\theta_0 = 0.02$), $D_{\text{KL}}$ is massive, and $\mathbb{E}[N]
\approx 1.2$ runs: SPRT terminates on the first or second retry, saving 60–70%
of CI compute over static retry policies!



## 4. The CI Pipeline as a Shannon Noisy Channel

Let us complete the unification by viewing continuous integration through Claude
Shannon's 1948 masterwork: **Communication in the Presence of Noise**.

Consider the CI runner as a **Binary Symmetric Channel (BSC)**:
- **Transmitter (Input $X$)**: The true code state ($X=0$ for Clean Code, $X=1$
  for Buggy Code).
- **Noise / Distortion**: Flakiness noise (race conditions, async timeouts,
  network jitter) that flips output bits with crossover probability $\epsilon$.
- **Receiver (Output $Y$)**: The test pass/fail assertion reported in the
  console.

```
       Code Commit (X) ---------> [ Noisy CI Pipeline (e) ] ---------> Test
Result (Y)
          X in {0, 1}                    p(y != x) = e                   Y in
{0, 1}
```

The **Channel Capacity** of a binary symmetric channel with crossover error
probability $\epsilon$ is:

$$
C = 1 - H_2(\epsilon) = 1 + \epsilon \log_2(\epsilon) + (1 - \epsilon) \log_2(1
- \epsilon)
$$


- If $\epsilon = 0$ (zero flakiness), $C = 1.0$ bit/run: every test execution
  transmits 1 perfect bit of truth about whether the code is broken.
- If $\epsilon = 0.5$ (total chaos / pure coin toss), $C = 0$ bits/run: no
  amount
  of test execution can ever reveal code correctness!
- If $\epsilon = 0.05$ ($5\%$ flakiness), $C = 1 - H_2(0.05) \approx 1 - 0.286 =
  0.714$ bits/run.

### Test Retries as Repeat-Accumulate Channel Codes
Why do engineers retry tests? Because they are unconsciously using a
**repetition code** of rate $R = 1 / N_{\text{retries}}$ to overcome channel
noise! Shannon's Channel Coding Theorem proves:

$$
\text{Reliable Merge Decision Possible} \iff R < C
$$


When an engineering org has flaky tests with high entropy $H_2(\epsilon)$, they
are forced to spend massive cloud compute repeating tests to operate below
channel capacity.



## 5. What Do We Actually Do With This Number?
### From Wasted Bits to Operational Action

Every concept in this chapter translates directly into an operational CI
pipeline engine.

### Action 1: The Adaptive Wald SPRT CI Retry Bot
- **The Problem**: Fixed 'Retry 3 times' policies waste tens of thousands of
cloud compute hours running pointless retries on clearly broken PRs, while
under-sampling ambiguous borderline failures.
- **The Operational Rule**: Replace static retries with a dynamic SPRT runner:
  1. Define $H_0: \theta \le 0.02$ (flakiness baseline) and $H_1: \theta \ge
     0.40$
(regression).
  2. Set target error bounds: $\alpha_{\text{err}} = 0.01$ (1% false alarm),
$\beta_{\text{err}} = 0.001$ (0.1% escape).
  3. After each individual test run $m$, compute $S_m$.
  4. If $S_m$ crosses boundary $A$ or $B$, **terminate the CI job immediately**.
  - **Result**: Average retries for clean builds drops from 3.0 to 1.1; average
retries for catastrophic bugs drops from 3.0 to 1.3. Overall CI compute savings:
**62%**.

### Action 2: Dynamic Cost-Weighted Merge Gating
- **The Problem**: Tier-1 payment checkout services cannot tolerate the same
risk as an internal admin documentation typo fix.
- **The Operational Rule**: Parameterize the merge threshold $p^*$ by service
SLA:

$$
p^*(s) = \frac{C_{\text{false\_block}}}{C_{\text{break}}(s) + C_{\text{false\_block}}}
$$

  - Tier-1 Service ($C_{\text{break}} = \$100{,}000, C_{\text{false\_block}} =
    \$100$):
$p^* \approx 0.10\%$ (requires overwhelming certainty before merging).
  - Monorepo Service ($C_{\text{break}} = \$2{,}000, C_{\text{false\_block}} =
    \$100$):
$p^* = 4.76\%$ (blocks whenever bug probability exceeds $4.76\%$).
  - Internal Tool ($C_{\text{break}} = \$300, C_{\text{false\_block}} = \$100$):
$p^* = 25.0\%$ (prioritizes developer velocity).

### Action 3: Fleet Channel Capacity Monitoring
- **The Problem**: Flakiness silently degrades CI efficiency over quarters as
tests accumulate technical debt.
- **The Operational Rule**: Compute the monthly fleet channel capacity
$\bar{C} = 1 - \frac{1}{J}\sum_j H_2(\hat{\epsilon}_j)$.
  - If fleet capacity drops below $0.85$ bits/run, trigger an automated
**'Flakiness Debt Sprint'** to refactor top-entropy tests before CI compute
costs double.



## 6. Summary and Transition

| Decision Framework | Primary Metric | Decision Boundary | Compute Efficiency |
| :--- | :--- | :--- | :--- |
| **Naive Static Retries** | Fixed count $N=3$ | Majority vote | Low (Wastes 60% compute on easy cases) |
| **Bayesian Decision Theory** | Expected Loss $R(a \mid D)$ | $p^* = \frac{C_{\text{false\_block}}}{C_{\text{break}} + C_{\text{false\_block}}} = 4.76\%$ | High (Aligns directly with business dollars) |
| **EVSI Re-Run Gate** | Net value of sampling | EVSI $> C_{\text{run}} = \$1.50 \implies [0.44\%, 7.43\%]$ | Optimal (Re-runs only when uncertainty pays) |
| **Wald Sequential Test (SPRT)** | Log-likelihood ratio $S_m$ | Boundaries $A, B$ based on $(\alpha, \beta)$ | **Optimal (Mathematically minimal samples)** |
| **Shannon Channel View** | Channel Capacity $C = 1 - H_2(\epsilon)$ | Repetition code rate $R < C$ | Quantifies maximum possible CI throughput |

In the next chapter (**Chapter 10: Active Learning & Experimental Design —
Expected Information Gain**), we move from passive observation to active
interrogation: *When an outage occurs across 1,000 commits, which exact test or
commit should we run next to extract maximum bits of diagnostic certainty?*

